# Transform - capa silver
Se limpian y se integran las tablas de bronze:
- `productos`: products + aisles + departments (overwrite)
- `ordenes`: orders (merge por order_id)
- `detalle_ordenes`: order_products + ordenes + productos (merge por order_id y product_id)

In [0]:
%run ./00_utils

In [0]:
from pyspark.sql.types import StringType

params = obtener_parametros()
CATALOGO = params["catalogo"]
BRONZE, SILVER = f"{CATALOGO}.bronze", f"{CATALOGO}.silver"

## Funciones

In [0]:
DIAS = {0: "Domingo", 1: "Lunes", 2: "Martes", 3: "Miércoles", 4: "Jueves", 5: "Viernes", 6: "Sábado"}

def nombre_dia(dow):
    return DIAS.get(dow, "Desconocido") if dow is not None else "Desconocido"

nombre_dia_udf = F.udf(nombre_dia, StringType())


def franja_horaria(col_hora):
    return (F.when(col_hora.between(0, 5), "Madrugada")
             .when(col_hora.between(6, 11), "Mañana")
             .when(col_hora.between(12, 17), "Tarde")
             .otherwise("Noche"))


# Se quitan las barras invertidas de escape y los espacios repetidos
def limpiar_texto(col_texto):
    sin_escapes = F.regexp_replace(col_texto, r"\\", "")
    return F.trim(F.regexp_replace(sin_escapes, r"\s+", " "))

## silver.productos

In [0]:
try:
    df_products = (spark.table(f"{BRONZE}.products")
                   .filter(F.col("product_id").isNotNull())
                   .dropDuplicates(["product_id"]))
    df_aisles = spark.table(f"{BRONZE}.aisles").dropDuplicates(["aisle_id"])
    df_departments = spark.table(f"{BRONZE}.departments").dropDuplicates(["department_id"])

    df_productos = (df_products.alias("p")
        .join(F.broadcast(df_aisles.alias("a")), F.col("p.aisle_id") == F.col("a.aisle_id"), "left")
        .join(F.broadcast(df_departments.alias("d")), F.col("p.department_id") == F.col("d.department_id"), "left")
        .select(
            F.col("p.product_id"),
            limpiar_texto(F.col("p.product_name")).alias("nombre_producto"),
            F.col("p.aisle_id"),
            F.initcap(F.coalesce(F.col("a.aisle"), F.lit("sin clasificar"))).alias("pasillo"),
            F.col("p.department_id"),
            F.initcap(F.coalesce(F.col("d.department"), F.lit("sin clasificar"))).alias("departamento"),
        )
        # En la fuente el valor 'missing' indica que no tiene clasificacion
        .withColumn("pasillo", F.when(F.lower("pasillo") == "missing", "Sin Clasificar").otherwise(F.col("pasillo")))
        .withColumn("departamento", F.when(F.lower("departamento") == "missing", "Sin Clasificar").otherwise(F.col("departamento")))
        .withColumn("es_organico", F.lower(F.col("nombre_producto")).contains("organic"))
        .withColumn("fecha_proceso", F.current_timestamp()))

    validar_no_vacio(df_productos, "silver.productos")
    escribir_tabla(df_productos, f"{SILVER}.productos", modo="overwrite")
except Exception as e:
    log(f"Error: {e}", "ERROR", "silver.productos")
    raise

## silver.ordenes

In [0]:
try:
    df_ordenes = (spark.table(f"{BRONZE}.orders")
        # Las ordenes 'test' no tienen detalle de productos
        .filter(F.col("eval_set").isin("prior", "train"))
        .filter(F.col("order_id").isNotNull() & F.col("user_id").isNotNull())
        .filter(F.col("order_dow").between(0, 6) & F.col("order_hour_of_day").between(0, 23))
        .dropDuplicates(["order_id"])
        .select(
            "order_id",
            "user_id",
            "eval_set",
            F.col("order_number").alias("numero_orden"),
            F.col("order_dow").alias("dia_semana_num"),
            nombre_dia_udf(F.col("order_dow")).alias("dia_semana"),
            F.col("order_hour_of_day").alias("hora_dia"),
            franja_horaria(F.col("order_hour_of_day")).alias("franja_horaria"),
            F.coalesce(F.col("days_since_prior_order"), F.lit(0)).cast("int").alias("dias_desde_orden_previa"),
            F.col("days_since_prior_order").isNull().alias("es_primera_orden"),
        )
        .withColumn("fecha_proceso", F.current_timestamp()))

    validar_no_vacio(df_ordenes, "silver.ordenes")
    merge_delta(df_ordenes, f"{SILVER}.ordenes", ["order_id"])
except Exception as e:
    log(f"Error: {e}", "ERROR", "silver.ordenes")
    raise

## silver.detalle_ordenes

In [0]:
try:
    df_op = (spark.table(f"{BRONZE}.order_products")
             .filter(F.col("order_id").isNotNull() & F.col("product_id").isNotNull())
             .dropDuplicates(["order_id", "product_id"]))
    df_ord = spark.table(f"{SILVER}.ordenes")
    df_prod = spark.table(f"{SILVER}.productos")

    # Se cuentan las lineas que no tienen su orden en la tabla ordenes
    total_op = df_op.count()
    huerfanos = df_op.join(df_ord, "order_id", "left_anti").count()
    pct = (huerfanos / total_op * 100) if total_op else 0
    log(f"{huerfanos:,} de {total_op:,} lineas ({pct:.1f}%) sin orden, se descartan",
        "WARN" if huerfanos else "INFO", "silver.detalle_ordenes")

    df_detalle = (df_op.alias("op")
        .join(df_ord.alias("o"), "order_id", "inner")
        .join(F.broadcast(df_prod.alias("p")), "product_id", "inner")
        .select(
            "order_id",
            "product_id",
            F.col("o.user_id"),
            F.col("op.add_to_cart_order").alias("orden_en_carrito"),
            (F.col("op.reordered") == 1).alias("es_recompra"),
            F.col("p.nombre_producto"),
            F.col("p.pasillo"),
            F.col("p.departamento"),
            F.col("p.es_organico"),
            F.col("o.numero_orden"),
            F.col("o.dia_semana_num"),
            F.col("o.dia_semana"),
            F.col("o.hora_dia"),
            F.col("o.franja_horaria"),
        )
        .withColumn("fecha_proceso", F.current_timestamp()))

    validar_no_vacio(df_detalle, "silver.detalle_ordenes")
    merge_delta(df_detalle, f"{SILVER}.detalle_ordenes", ["order_id", "product_id"])
except Exception as e:
    log(f"Error: {e}", "ERROR", "silver.detalle_ordenes")
    raise

In [0]:
display(spark.table(f"{SILVER}.detalle_ordenes").limit(10))